# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [11]:
# Only needed for Udacity workspace

import importlib.util
import sys


In [12]:
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
import chromadb
from pydantic import BaseModel
from tavily import TavilyClient
import json

In [13]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


In [14]:
chroma_client = chromadb.PersistentClient(path=".data/chromadb")
embedding_fn = chromadb.utils.embedding_functions.OpenAIEmbeddingFunction()
collection = chroma_client.get_collection("games")


#### Retrieve Game Tool

In [15]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# Tool Docstring:
#    

@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game"""
    collection = chroma_client.get_collection("games")

    results = collection.query(query_texts=[query], n_results=5)
    print(results)
    return results

#### Evaluate Retrieval Tool

In [16]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str
    
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result"""
    llm = LLM(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
    Question: {question}
    Retrieved Docs: {retrieved_docs}
    Give a detailed explanation, so it's possible to take an action to accept it or not.
    """
    result = llm.invoke(prompt, response_format=EvaluationReport)
    return result.dict()


#### Game Web Search Tool

In [17]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def web_search(query: str) -> str:
    """Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry."""
    results = client.search(query)
    # Return as a JSON string so the LLM can read the structured results
    print(f"Tool 'web_search' called with query={query!r}, results={results!r}")
    return json.dumps(results, indent=2)

### Agent

In [18]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
insructions = """You are an assistant for a videogame company. Your task is to answer questions about the videogame industry.
To answer the questions, you can use the following tools:
- retrieve_game: use this tool to search in the videogame vector database. You can use it when you need to find specific information about videogames, like release dates, platforms, sales, etc. 
- evaluate_retrieval: use this tool to evaluate if the retrieved documents are useful to answer the user's question. You can use it after retrieve_game to check if the retrieved documents are good enough to answer the question. 
- web_search: use this tool to search the web for game content. Only use this tool when evaluate_retrieval indicates that the retrieved documents are not sufficient.
"""
agent = Agent(model_name="gpt-4o-mini", instructions=insructions, tools=[retrieve_game, evaluate_retrieval, web_search])

In [19]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
for i, question in enumerate([
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?"
]):
    print("\n\n")
    print("=" * 80)
    print(f"Question {i+1}: {question}")
    run = agent.invoke(question)
    print(f"Answer {i+1}: {run.get_final_state()["messages"][-1].content}")

    print("\nMessage Chain:")    
    for message in run.get_final_state()["messages"]:
        if isinstance(message, ToolMessage):
            print(f"Tool Result: {message.content}")
        elif isinstance(message, AIMessage) and message.tool_calls:
            print(f"Tool Call Message: {message.tool_calls}")
        else:
            print(f"Message: {message.content}")
    print(f"\nfull trace: {run.get_final_state()}")




Question 1: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
{'ids': [['006', '007', '012', '009', '008']], 'embeddings': None, 'documents': [['[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', '[Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.', '[Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.', "[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", '[Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest

### (Optional) Advanced

In [20]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes